# Resistivity change (%)

In [5]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colorbar import ColorbarBase
from scipy.interpolate import griddata

batches = {
    "batch1": {
        "files": {
            "February 25": "f008_diffres.dat",
            "March 7":     "f015_diffres.dat",
            "March 17":    "f022_diffres.dat",
            "March 27":    "f024_diffres.dat",
        },
        "norm":     mcolors.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100),
        "ticks":    [-100, -75, -50, -25, 0, 25, 50, 75, 100],
        "out":      "ERT_batch1_Feb25_Mar27.png",
    },
    "batch2": {
        "files": {
            "March 31":  "f026_diffres.dat",
            "April 6":   "f028_diffres.dat",
            "April 10":  "f030_diffres.dat",
        },
        "norm":     mcolors.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100),
        "ticks":    [-100, -75, -50, -25, 0, 25, 50, 75, 100],
        "out":      "ERT_batch2_Mar31_Apr10.png",
    },
    "batch3": {
        "files": {
            "April 17":  "f038_diffres.dat",
            "April 19":  "f040_diffres.dat",
            "April 20":  "f044_diffres.dat",
        },
        "norm":     mcolors.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100),
        "ticks":    [-100, -75, -50, -25, 0, 25, 50, 75, 100],
        "out":      "ERT_batch3_Apr17_Apr20.png",
    },
}

cmap_ert = plt.cm.RdBu_r
xi = np.linspace(10, 27, 1400)
zi = np.linspace(-1.25, 0.0, 350)
XI, ZI = np.meshgrid(xi, zi)


def make_figure(files: dict, norm, cb_ticks: list, out_png: str) -> None:
    n = len(files)
    fig, axes = plt.subplots(
        n, 1, figsize=(20, n * 3.5),
        gridspec_kw={'hspace': 0.75, 'left': 0.07, 'right': 0.88,
                     'top': 0.96, 'bottom': 0.07}
    )
    if n == 1:
        axes = [axes]
    fig.patch.set_facecolor('white')

    vmin = norm.vmin
    vmax = norm.vmax

    for i, (ax, (date, fpath)) in enumerate(zip(axes, files.items())):
        d = np.loadtxt(fpath)
        mask = (d[:, 0] >= 9.5) & (d[:, 0] <= 27.5) & \
               (d[:, 1] >= -1.35) & (d[:, 1] <= 0.05)
        x, z, diff_pct = d[mask, 0], d[mask, 1], d[mask, 2]

        grid_lin = griddata((x, z), diff_pct, (XI, ZI), method='linear')
        grid_nn  = griddata((x, z), diff_pct, (XI, ZI), method='nearest')
        grid = np.where(np.isnan(grid_lin), grid_nn, grid_lin)
        grid = np.clip(grid, vmin, vmax)

        ax.set_facecolor('white')
        levels_fill = np.linspace(vmin, vmax, 101)
        ax.contourf(XI, ZI, grid, levels=levels_fill,
                    cmap=cmap_ert, norm=norm, extend='both')
        ax.contour(XI, ZI, grid, levels=14,
                   colors='black', linewidths=0.5, alpha=0.35)

        ax.set_xlim(10, 27)
        ax.set_ylim(-1.25, 0.0)

        for gx in [16.0, 21.5]:
            ax.axvline(gx, color='grey', linestyle='--', linewidth=1.4, alpha=0.9)

        if i == 0:
            for label, lx in [("IPLR-760", 13.0), ("WM-140", 18.75), ("WM-203", 24.25)]:
                ax.text(lx, 0.15, label, color='red', fontsize=20,
                        fontweight='bold', ha='center', va='bottom',
                        transform=ax.transData, clip_on=False)

        ax.text(10.15, -0.03, date, color='black', fontsize=18,
                va='top', ha='left', style='italic', transform=ax.transData)

        ax.tick_params(colors='black', labelsize=18)
        for spine in ax.spines.values():
            spine.set_edgecolor('#aaaaaa')
        ax.set_ylabel("Depth (m)", color='black', fontsize=20)
        ax.set_yticks([-1.0, -0.5, 0.0])
        ax.set_yticklabels(['-1.0', '-0.5', '0'], color='black', fontsize=18)

        ax.set_xticks(range(10, 28, 2))
        if i == n - 1:
            ax.set_xlabel("Distance (m)", color='black', fontsize=20)
            ax.set_xticklabels([str(v) for v in range(10, 28, 2)],
                               color='black', fontsize=18)
        else:
            ax.set_xticklabels([])

    # ── Colorbar ──────────────────────────────────────────────────────────────
    cbar_ax = fig.add_axes([0.905, 0.07, 0.018, 0.89])
    cb = ColorbarBase(cbar_ax, cmap=cmap_ert, norm=norm, orientation='vertical')
    cb.set_ticks(cb_ticks)
    cb.ax.set_yticklabels([str(t) for t in cb_ticks])
    cb.ax.tick_params(colors='black', labelsize=18)
    cb.set_label('Resistivity change (%)', color='black', fontsize=20,
                 rotation=270, labelpad=26)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='black', fontsize=18)
    for spine in cb.ax.spines.values():
        spine.set_edgecolor('#aaaaaa')

    fig.savefig(out_png, dpi=600, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f"Saved: {out_png}")
    plt.close(fig)


for b in batches.values():
    make_figure(b["files"], b["norm"], b["ticks"], b["out"])

Saved: ERT_batch1_Feb25_Mar27.png
Saved: ERT_batch2_Mar31_Apr10.png
Saved: ERT_batch3_Apr17_Apr20.png
